# 小样本平面图识别：基座模型 + LoRA 微调
## SAM2 / DINOv2 + LoRA + 多任务头（200张图纸场景）

```
架构总览
─────────────────────────────────────────────────────────
输入图片 (512×512)
    │
    ▼
基座编码器 (SAM2-Hiera / DINOv2-ViT-L)
  └─ 冻结原始权重  95%+
  └─ LoRA 低秩矩阵  r=16  注入 Self-Attention
    │
    ▼
特征金字塔 FPN  (256ch)
  ├─ 分割头  →  wall_mask  (BCE + Dice + Affinity)
  └─ 检测头  →  door/window boxes  (L1 + GIoU)
    │
    ▼
OpenCV 硬逻辑后处理
  Opening → Closing → Shrinking → Overlap 消解
    │
    ▼
Staging → VLM 质检 → 人工审核 → 生产库
─────────────────────────────────────────────────────────
```

### 为什么换掉 ResNet50？
| 维度 | ResNet50（原方案） | SAM2/DINOv2（本方案） |
|---|---|---|
| 预训练数据 | ImageNet 1.2M | SA-1B 11M+ / LVD-142M |
| 几何感知 | 弱（分类导向） | 强（分割/自监督导向） |
| 200样本微调 | 易过拟合 | LoRA 仅训练 ~1% 参数 |
| 直角/线段理解 | 需从头学 | 预训练已内化 |

### 依赖安装
```bash
pip install torch torchvision
pip install segment-anything-2   # SAM2
pip install timm                 # DINOv2
pip install loralib              # LoRA 层
pip install albumentations       # 数据增强
pip install mlflow
```

## 0. 环境配置

In [20]:
import warnings
warnings.filterwarnings('ignore')

import os, sys, gc, json, time, random, logging
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Tuple, Optional, Dict
from collections import OrderedDict

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.ops import FeaturePyramidNetwork, generalized_box_iou_loss
import albumentations as A
from albumentations.pytorch import ToTensorV2
import mlflow

# ── 可选依赖（按需加载）──
try:
    import loralib as lora
    HAS_LORA = True
except ImportError:
    HAS_LORA = False
    print('[警告] loralib 未安装: pip install loralib')

try:
    import timm
    HAS_TIMM = True
except ImportError:
    HAS_TIMM = False
    print('[警告] timm 未安装（DINOv2 需要）: pip install timm')

# 或者直接手动指定你的 sam2 源码路径
sam2_path = "/workspace/segment-anything-2" 
post_path = "/workspace/production_3d/postfile"
pre_path = "/workspace/production_3d/prefile"

if sam2_path and post_path and pre_path not in sys.path:
    sys.path.append(sam2_path)
    sys.path.append(post_path)
    sys.path.append(pre_path)


try:
    from sam2.build_sam import build_sam2
    from sam2.modeling.sam2_base import SAM2Base
    HAS_SAM2 = True
except ImportError:
    HAS_SAM2 = False
    print('[警告] SAM2 未安装: pip install segment-anything-2')

# ── 路径配置 ──
if os.path.exists('/workspace/production_3d'):
    BASE_DIR       = '/workspace/production_3d'
    CUBICASA_ROOT  = '/workspace/CubiCasa5k'
    DATA_FOLDER    = '/workspace/data/cubicasa5k/'
    COMPANY_DATA   = '/workspace/data/company_floorplans/'   # 200张公司图纸
    SAM2_CKPT      = '/workspace/checkpoints/sam2_hiera_large.pt'
elif os.path.exists('/content'):
    BASE_DIR       = '/content'
    CUBICASA_ROOT  = '/content/CubiCasa5k'
    DATA_FOLDER    = '/content/data/cubicasa5k/'
    COMPANY_DATA   = '/content/data/company_floorplans/'
    SAM2_CKPT      = '/content/checkpoints/sam2_hiera_large.pt'
else:
    BASE_DIR       = '.'
    CUBICASA_ROOT  = r'E:\JOB\CubiCasa5k'
    DATA_FOLDER    = r'C:/Users/kawayi_yaling/.cache/kagglehub/datasets/qmarva/cubicasa5k/versions/4/cubicasa5k/cubicasa5k/'
    COMPANY_DATA   = './data/company_floorplans/'
    SAM2_CKPT      = './checkpoints/sam2_hiera_large.pt'

CHECKPOINT_DIR = os.path.join(BASE_DIR, 'checkpoints_lora')
MLFLOW_DIR     = os.path.join(BASE_DIR, 'mlruns')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(COMPANY_DATA, exist_ok=True)

sys.path.insert(0, CUBICASA_ROOT)
sys.path.insert(0, BASE_DIR)
os.chdir(CUBICASA_ROOT)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S',
)
logger = logging.getLogger(__name__)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('✓ 环境配置完成')
print(f'  device    : {DEVICE}')
print(f'  HAS_SAM2  : {HAS_SAM2}')
print(f'  HAS_TIMM  : {HAS_TIMM}')
print(f'  HAS_LORA  : {HAS_LORA}')
if torch.cuda.is_available():
    print(f'  GPU       : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM      : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

✓ 环境配置完成
  device    : cuda
  HAS_SAM2  : True
  HAS_TIMM  : True
  HAS_LORA  : True
  GPU       : NVIDIA RTX A6000
  VRAM      : 50.9 GB


## 1. 训练配置（LoRA + 多任务）

In [21]:
@dataclass
class LoRAFloorplanConfig:
    """
    LoRA 微调配置
    200 张图纸场景的专项设置
    """
    # ── 基座模型选择 ──
    # 'sam2'     : SAM2-Hiera-Large（首推，分割专用预训练）
    # 'dinov2'   : DINOv2-ViT-L/14（次选，自监督视觉表征）
    backbone_type:     str  = 'sam2'
    # SAM2 yaml 由 SAM2EncoderWrapper 内部根据 model_size 自动选择
    # 不需要手动指定路径（yaml 随 SAM2 包一起安装）
    sam2_model_size:   str  = 'large'   # 'tiny'|'small'|'base'|'large'
    dinov2_model:      str  = 'vit_large_patch14_dinov2'  # timm 模型名

    # ── LoRA 超参数 ──
    lora_r:            int   = 16     # 低秩矩阵的秩 r
    lora_alpha:        int   = 32     # 缩放因子 = alpha/r，控制 LoRA 贡献强度
    lora_dropout:      float = 0.1    # LoRA 层的 dropout，防止小样本过拟合
    # 注入 LoRA 的层（Self-Attention 的 Q/V 投影）
    lora_target_modules: List[str] = field(
        default_factory=lambda: ['q_proj', 'v_proj']
    )
    # 冻结策略：只有 LoRA 参数 + 任务头 参与训练
    freeze_backbone:   bool  = True

    # ── FPN / 任务头 ──
    fpn_out_channels:  int   = 256
    seg_num_classes:   int   = 2      # background=0, wall=1
    det_num_classes:   int   = 3      # background=0, door=1, window=2

    # ── 数据（200张公司图纸）──
    company_data_dir:  str   = COMPANY_DATA
    # 同时也可以混入 CubiCasa（迁移学习 → 领域适应）
    use_cubicasa_mix:  bool  = True
    cubicasa_mix_ratio: float = 0.3   # 每个 batch 中 CubiCasa 样本占 30%
    cubicasa_folder:   str   = DATA_FOLDER

    # ── 图片规格 ──
    wall_class_id:     int   = 2
    door_class_id:     int   = 2
    window_class_id:   int   = 1
    min_bbox_area:     int   = 100
    crop_to_wall:      bool  = True
    crop_padding:      int   = 10
    min_crop_size:     int   = 64

    # ── 滑动窗口 / Tiling ──
    tile_size:         int   = 512
    tile_overlap:      int   = 64

    # ── 极致数据增强（小样本必须）──
    aug_rotation_degrees: List[int] = field(
        default_factory=lambda: [0, 90, 180, 270]
    )
    aug_scale_range:   Tuple[float, float] = (0.5, 2.0)
    aug_use_flip:      bool  = True
    # 合成扰动（模拟扫描图纸噪声）
    aug_add_noise:     bool  = True
    aug_noise_var:     float = 0.02   # 高斯噪声方差
    aug_fake_lines:    bool  = True   # 随机叠加虚假线条
    aug_jpeg_quality:  Tuple[int, int] = (60, 95)  # 模拟扫描质量

    # ── 归一化 ──
    norm_mean:         Tuple = (0.485, 0.456, 0.406)
    norm_std:          Tuple = (0.229, 0.224, 0.225)

    # ── 训练超参数 ──
    batch_size:        int   = 4
    num_workers:       int   = 2
    max_epochs:        int   = 60     # 小样本多训一些 epoch
    learning_rate:     float = 5e-5   # LoRA 比全量训练用更小的 lr
    weight_decay:      float = 1e-4
    warmup_epochs:     int   = 5
    use_amp:           bool  = True
    grad_clip:         float = 3.0

    # ── 损失函数权重（动态 lambda）──
    # L_total = lambda_seg * L_seg + lambda_det * L_det
    lambda_seg:        float = 1.0
    lambda_det:        float = 1.0
    use_dynamic_lambda: bool = True   # 开启动态 lambda 平衡
    # Dice Loss 权重（相比纯 BCE，Dice 对小目标更友好）
    dice_weight:       float = 0.5
    bce_weight:        float = 0.5
    wall_class_weight: float = 5.0
    # GIoU 检测损失
    giou_weight:       float = 2.0
    l1_weight:         float = 5.0

    # ── 检查点 ──
    checkpoint_dir:    str   = CHECKPOINT_DIR
    save_top_k:        int   = 3

    # ── MLflow ──
    mlflow_experiment: str   = 'floorplan_lora'
    mlflow_run_name:   str   = f'lora_{time.strftime("%Y%m%d_%H%M%S")}'


CFG = LoRAFloorplanConfig()
Path(CFG.checkpoint_dir).mkdir(parents=True, exist_ok=True)

print('✓ LoRA 训练配置完成')
print(f'  backbone    : {CFG.backbone_type}')
print(f'  lora_r      : {CFG.lora_r}  alpha={CFG.lora_alpha}')
print(f'  epochs      : {CFG.max_epochs}  lr={CFG.learning_rate}')
print(f'  tile_size   : {CFG.tile_size}  batch={CFG.batch_size}')
print(f'  aug_noise   : {CFG.aug_add_noise}  fake_lines={CFG.aug_fake_lines}')
print(f'  dynamic_λ   : {CFG.use_dynamic_lambda}')

✓ LoRA 训练配置完成
  backbone    : sam2
  lora_r      : 16  alpha=32
  epochs      : 60  lr=5e-05
  tile_size   : 512  batch=4
  aug_noise   : True  fake_lines=True
  dynamic_λ   : True


## 2. Step 1：基座模型加载（SAM2 / DINOv2）

In [22]:
# ══════════════════════════════════════════════════════════════
# SAM2 Hiera 编码器包装
# ══════════════════════════════════════════════════════════════

class SAM2EncoderWrapper(nn.Module):
    """
    把 SAM2 的图像编码器（Hiera ViT）包装成标准的特征提取接口
    输出 4 个尺度的特征图，供 FPN 使用

    SAM2 Hiera 天然支持多尺度输出（设计用于分割），
    比 ViT-Plain 更适合 FPN 接头。
    """

    # SAM2 yaml 配置名称对应表（仓库 sam2/configs/ 目录下的文件名）
    CONFIG_MAP = {
        'tiny':   'sam2_hiera_t.yaml',
        'small':  'sam2_hiera_s.yaml',
        'base':   'sam2_hiera_b+.yaml',
        'large':  'sam2_hiera_l.yaml',   # 推荐
    }

    def __init__(self, sam2_ckpt: str, model_size: str = 'large'):
        """
        sam2_ckpt  : .pt 权重文件路径
                     下载: wget https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_large.pt
        model_size : 'tiny' | 'small' | 'base' | 'large'
                     yaml 文件由 SAM2 包自动找到，不需要手动指定路径
        """
        super().__init__()
        if not HAS_SAM2:
            raise RuntimeError('请安装 SAM2: pip install segment-anything-2')

        sam2_cfg = self.CONFIG_MAP.get(model_size, 'sam2_hiera_l.yaml')

        # build_sam2 第一个参数是 yaml 文件名（相对于 SAM2 包的 configs/ 目录）
        # SAM2 内部用 Hydra 从包内自动定位，不需要绝对路径
        sam2_model = build_sam2(sam2_cfg, sam2_ckpt, device='cpu')
        self.encoder = sam2_model.image_encoder

        # SAM2 Hiera-Large 的多尺度输出通道数（stride: 4, 8, 16, 32）
        channels_map = {
            'tiny':  [96,  192,  384,  768],
            'small': [96,  192,  384,  768],
            'base':  [112, 224,  448,  896],
            'large': [144, 288,  576, 1152],
        }
        self.out_channels = channels_map.get(model_size, [144, 288, 576, 1152])

    def forward(self, x: torch.Tensor) -> OrderedDict:
        """
        x: (B, 3, H, W)
        返回 OrderedDict {'0':c1, '1':c2, '2':c3, '3':c4}
        供 FPN 使用
        """
        # SAM2 image_encoder 返回多尺度特征
        # backbone_out 包含 'vision_features' 和 'vision_pos_enc'
        backbone_out = self.encoder(x)
        feats = backbone_out.get('backbone_fpn', [])

        # 取 4 个尺度，包装成 FPN 期待的 OrderedDict
        if len(feats) >= 4:
            return OrderedDict([
                ('0', feats[0]),
                ('1', feats[1]),
                ('2', feats[2]),
                ('3', feats[3]),
            ])
        # 特征数不足时：重复最后一个
        result = OrderedDict()
        for i in range(4):
            idx = min(i, len(feats) - 1)
            result[str(i)] = feats[idx]
        return result


# ══════════════════════════════════════════════════════════════
# DINOv2 ViT 编码器包装
# ══════════════════════════════════════════════════════════════

class DINOv2EncoderWrapper(nn.Module):
    """
    把 DINOv2 ViT-L/14 包装成 FPN 可用的多尺度特征提取器

    DINOv2 是 plain ViT，无原生多尺度输出。
    我们从不同深度的 Transformer block 抽取中间特征作为多尺度。
    """

    # ViT-L: 24 个 block，embed_dim=1024
    EXTRACT_LAYERS = [5, 11, 17, 23]   # 对应 1/4 深度点

    def __init__(self, model_name: str = 'vit_large_patch14_dinov2'):
        super().__init__()
        if not HAS_TIMM:
            raise RuntimeError('请安装 timm: pip install timm')

        self.vit = timm.create_model(
            model_name,
            pretrained=True,
            features_only=False,  # plain ViT 不支持 features_only
        )
        embed_dim = self.vit.embed_dim   # ViT-L: 1024

        # 用 1×1 卷积把各中间层的维度统一为 FPN 输入需要的通道数
        # 逐层通道数相同（都是 embed_dim），但空间分辨率不同
        self.out_channels = [embed_dim] * 4

        # 注册 hook 钩子，抽取中间层特征
        self._features: Dict[int, torch.Tensor] = {}
        for layer_idx in self.EXTRACT_LAYERS:
            self.vit.blocks[layer_idx].register_forward_hook(
                self._make_hook(layer_idx)
            )

    def _make_hook(self, layer_idx: int):
        def hook(module, input, output):
            self._features[layer_idx] = output
        return hook

    def forward(self, x: torch.Tensor) -> OrderedDict:
        B, C, H, W = x.shape
        self._features.clear()

        _ = self.vit(x)  # 触发 hook

        patch_h = H // 14
        patch_w = W // 14
        result  = OrderedDict()

        for i, layer_idx in enumerate(self.EXTRACT_LAYERS):
            feat = self._features[layer_idx]  # (B, N_patches+1, embed_dim)
            # 去掉 CLS token，reshape 成空间特征图
            feat = feat[:, 1:, :].permute(0, 2, 1)  # (B, embed_dim, N)
            feat = feat.reshape(B, -1, patch_h, patch_w)  # (B, D, pH, pW)
            # 对不同层做不同倍率的下采样，模拟多尺度
            if i < len(self.EXTRACT_LAYERS) - 1:
                scale = 2 ** (len(self.EXTRACT_LAYERS) - 1 - i)
                feat = F.avg_pool2d(feat, kernel_size=scale, stride=scale)
            result[str(i)] = feat

        return result


def build_backbone(cfg: LoRAFloorplanConfig) -> nn.Module:
    """
    根据配置选择 SAM2 或 DINOv2 骨干

    SAM2  加载 = yaml名称（SAM2包内自带）+ .pt权重（手动下载）
    DINOv2加载 = timm模型名（架构硬编码）+ HuggingFace权重（自动下载）
    """
    if cfg.backbone_type == 'sam2':
        if not os.path.exists(SAM2_CKPT):
            logger.warning(f'SAM2 权重不存在: {SAM2_CKPT}')
            logger.warning('下载命令: wget https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_large.pt')
            logger.warning('回退到 DINOv2...')
            return DINOv2EncoderWrapper(cfg.dinov2_model)
        return SAM2EncoderWrapper(SAM2_CKPT, model_size='large')
    else:
        return DINOv2EncoderWrapper(cfg.dinov2_model)


print('✓ 基座编码器类定义完成')
print('  SAM2EncoderWrapper   : 多尺度 Hiera 特征 → FPN')
print('  DINOv2EncoderWrapper : 中间层 hook 抽取 → FPN')
print('  build_backbone(cfg)  : 根据 backbone_type 自动选择')

✓ 基座编码器类定义完成
  SAM2EncoderWrapper   : 多尺度 Hiera 特征 → FPN
  DINOv2EncoderWrapper : 中间层 hook 抽取 → FPN
  build_backbone(cfg)  : 根据 backbone_type 自动选择


## 3. Step 2：LoRA 注入（Self-Attention 低秩矩阵）

In [23]:
# ══════════════════════════════════════════════════════════════
# LoRA 线性层（手写版，不依赖 loralib）
# loralib 已安装时，也可以直接用 lora.Linear
# ══════════════════════════════════════════════════════════════

class LoRALinear(nn.Module):
    """
    在原始 Linear 层旁边并联一对低秩矩阵

    原始权重 W (d_in × d_out) 被冻结。
    LoRA 增量：ΔW = B · A  其中 A ∈ R^{r×d_in}, B ∈ R^{d_out×r}
    前向：y = xW^T + (xA^T)B^T · (alpha/r)

    只有 A 和 B 参与梯度更新，参数量从 d_in×d_out 降到 r×(d_in+d_out)
    """

    def __init__(
        self,
        original_linear: nn.Linear,
        r:       int   = 16,
        alpha:   int   = 32,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.original = original_linear
        self.r        = r
        self.scaling  = alpha / r

        d_in  = original_linear.in_features
        d_out = original_linear.out_features

        # 低秩矩阵：A 初始化为高斯，B 初始化为零（保证初始增量为 0）
        self.lora_A   = nn.Parameter(torch.randn(r, d_in)  * 0.02)
        self.lora_B   = nn.Parameter(torch.zeros(d_out, r))
        self.dropout  = nn.Dropout(dropout)

        # 冻结原始权重
        for param in self.original.parameters():
            param.requires_grad = False

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # 原始路径（冻结，不走梯度）
        base_out = self.original(x)
        # LoRA 增量路径
        lora_out = self.dropout(x) @ self.lora_A.T @ self.lora_B.T
        return base_out + lora_out * self.scaling


def inject_lora(
    model:           nn.Module,
    target_modules:  List[str],
    r:               int   = 16,
    alpha:           int   = 32,
    dropout:         float = 0.1,
) -> Tuple[nn.Module, int, int]:
    """
    遍历 model 中所有名称匹配 target_modules 的 Linear 层，
    将其替换为 LoRALinear。

    返回：(model, trainable_params, total_params)
    """
    replaced = 0

    def _replace_linear(parent: nn.Module, prefix: str):
        nonlocal replaced
        for name, child in list(parent.named_children()):
            full_name = f'{prefix}.{name}' if prefix else name
            if isinstance(child, nn.Linear) and any(
                t in name for t in target_modules
            ):
                lora_layer = LoRALinear(child, r=r, alpha=alpha, dropout=dropout)
                setattr(parent, name, lora_layer)
                replaced += 1
            else:
                _replace_linear(child, full_name)

    _replace_linear(model, '')

    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

    logger.info(
        f'LoRA 注入完成  替换了 {replaced} 个线性层  '
        f'可训练参数: {trainable/1e6:.2f}M / {total/1e6:.1f}M '
        f'({trainable/total*100:.1f}%)'
    )
    return model, trainable, total


def freeze_backbone(backbone: nn.Module) -> None:
    """冻结 backbone 的所有原始参数（LoRA 参数除外）"""
    for name, param in backbone.named_parameters():
        if 'lora_A' not in name and 'lora_B' not in name:
            param.requires_grad = False
    frozen    = sum(p.numel() for p in backbone.parameters() if not p.requires_grad)
    trainable = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
    logger.info(f'Backbone 冻结完成  冻结={frozen/1e6:.1f}M  活跃={trainable/1e6:.2f}M')


# ── 参数分组：给 LoRA 层和任务头设置不同的学习率 ──
def get_param_groups(model: nn.Module, base_lr: float) -> List[dict]:
    """
    学习率分层策略：
    - LoRA 参数     : base_lr（已经很小，不用进一步缩放）
    - 任务头参数    : base_lr * 5（头部需要快速适应新数据）
    - 其他可训练参数: base_lr
    """
    lora_params  = []
    head_params  = []
    other_params = []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if 'lora_A' in name or 'lora_B' in name:
            lora_params.append(param)
        elif 'seg_head' in name or 'det_fpn' in name or 'rpn' in name or 'roi_heads' in name:
            head_params.append(param)
        else:
            other_params.append(param)

    logger.info(
        f'参数分组  lora={len(lora_params)}  '
        f'head={len(head_params)}  other={len(other_params)}'
    )
    return [
        {'params': lora_params,  'lr': base_lr,       'name': 'lora'},
        {'params': head_params,  'lr': base_lr * 5.0, 'name': 'head'},
        {'params': other_params, 'lr': base_lr,       'name': 'other'},
    ]


print('✓ LoRA 注入工具函数定义完成')
print(f'  LoRALinear(r={16}, alpha={32}) : 可训练参数 ≈ {16}×(d_in+d_out)')
print('  inject_lora(model, ["q_proj","v_proj"]) : 一键注入')
print('  get_param_groups : LoRA × 1.0  任务头 × 5.0')

✓ LoRA 注入工具函数定义完成
  LoRALinear(r=16, alpha=32) : 可训练参数 ≈ 16×(d_in+d_out)
  inject_lora(model, ["q_proj","v_proj"]) : 一键注入
  get_param_groups : LoRA × 1.0  任务头 × 5.0


## 4. Step 1+2：组装完整模型

In [24]:
from torchvision.models.detection.rpn import (
    AnchorGenerator, RPNHead, RegionProposalNetwork
)
from torchvision.models.detection.roi_heads import RoIHeads
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.ops import MultiScaleRoIAlign
from torchvision.models.detection.image_list import ImageList


class SegmentationHead(nn.Module):
    """FPN + 轻量 decoder → wall mask（沿用原有设计）"""

    def __init__(self, in_channels_list: List[int],
                 fpn_out_channels: int = 256, num_classes: int = 2):
        super().__init__()
        self.fpn = FeaturePyramidNetwork(
            in_channels_list=in_channels_list,
            out_channels=fpn_out_channels,
        )
        self.decoder = nn.Sequential(
            nn.Conv2d(fpn_out_channels, 128, 3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 64, 3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, num_classes, 1),
        )

    def forward(self, features: OrderedDict, input_size: Tuple) -> torch.Tensor:
        fpn_out     = self.fpn(features)
        target_size = fpn_out['0'].shape[-2:]
        fused       = fpn_out['0']
        for k in ['1', '2', '3']:
            if k in fpn_out:
                fused = fused + F.interpolate(
                    fpn_out[k], size=target_size, mode='bilinear', align_corners=False
                )
        logits = self.decoder(fused)
        return F.interpolate(logits, size=input_size, mode='bilinear', align_corners=False)


class LoRAFloorplanModel(nn.Module):
    """
    LoRA 微调版平面图模型

    = 基座编码器（SAM2/DINOv2，95%+ 冻结）
    + LoRA 低秩矩阵（注入 Q/V 投影，r=16）
    + FPN 分割头（wall_mask）
    + Faster R-CNN 检测头（door/window boxes）
    """

    def __init__(self, cfg: LoRAFloorplanConfig):
        super().__init__()
        self.cfg = cfg

        # ── Step 1: 基座编码器 ──
        self.backbone = build_backbone(cfg)
        in_channels   = self.backbone.out_channels  # 各尺度通道数

        # ── Step 2: LoRA 注入 ──
        if cfg.freeze_backbone:
            freeze_backbone(self.backbone)
        self.backbone, self._trainable, self._total = inject_lora(
            self.backbone,
            target_modules  = cfg.lora_target_modules,
            r               = cfg.lora_r,
            alpha           = cfg.lora_alpha,
            dropout         = cfg.lora_dropout,
        )

        # ── 分割头 ──
        self.seg_head = SegmentationHead(
            in_channels_list = in_channels,
            fpn_out_channels = cfg.fpn_out_channels,
            num_classes      = cfg.seg_num_classes,
        )

        # ── 检测头（Faster R-CNN 风格）──
        fpn_ch = cfg.fpn_out_channels
        self.det_fpn = FeaturePyramidNetwork(
            in_channels_list = in_channels,
            out_channels     = fpn_ch,
        )
        rpn_anchor = AnchorGenerator(
            sizes           = ((16,), (32,), (64,), (128,)),
            aspect_ratios   = ((0.5, 1.0, 2.0),) * 4,
        )
        rpn_head = RPNHead(fpn_ch, rpn_anchor.num_anchors_per_location()[0])
        self.rpn = RegionProposalNetwork(
            rpn_anchor, rpn_head,
            fg_iou_thresh=0.7, bg_iou_thresh=0.3,
            batch_size_per_image=256, positive_fraction=0.5,
            pre_nms_top_n={'training': 2000, 'testing': 1000},
            post_nms_top_n={'training': 2000, 'testing': 300},
            nms_thresh=0.7,
        )
        box_roi_pool = MultiScaleRoIAlign(
            featmap_names=['0','1','2','3'], output_size=7, sampling_ratio=2
        )
        box_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(fpn_ch * 49, 1024), nn.ReLU(inplace=True),
            nn.Linear(1024, 1024),         nn.ReLU(inplace=True),
        )
        self.roi_heads = RoIHeads(
            box_roi_pool, box_head, FastRCNNPredictor(1024, cfg.det_num_classes),
            fg_iou_thresh=0.5, bg_iou_thresh=0.5,
            batch_size_per_image=512, positive_fraction=0.25,
            bbox_reg_weights=None,
            score_thresh=0.05, nms_thresh=0.5, detections_per_img=100,
        )

    def forward(self, images: torch.Tensor,
                targets: Optional[List[dict]] = None) -> dict:
        input_size   = images.shape[-2:]
        features     = self.backbone(images)
        seg_logits   = self.seg_head(features, input_size)
        det_features = self.det_fpn(features)
        image_sizes  = [images.shape[-2:]] * images.shape[0]
        img_list     = ImageList(images, image_sizes)

        if self.training and targets is not None:
            proposals, rpn_losses = self.rpn(img_list, det_features, targets)
            _, roi_losses = self.roi_heads(det_features, proposals, image_sizes, targets)
            return {'seg_logits': seg_logits,
                    'det_losses': {**rpn_losses, **roi_losses}}
        else:
            proposals, _ = self.rpn(img_list, det_features, None)
            det_outputs, _ = self.roi_heads(det_features, proposals, image_sizes, None)
            return {'seg_logits': seg_logits, 'det_outputs': det_outputs}


# ── 演示：模型参数统计（不实际加载基座权重）──
print('✓ LoRAFloorplanModel 定义完成')
print()
print('初始化示例（实际训练时运行）：')
print('  model = LoRAFloorplanModel(CFG).to(DEVICE)')
print('  # 输出：LoRA 注入完成  可训练参数: ~2.1M / ~308.5M (0.7%)')
print()
print('参数效率对比：')
print('  全量微调 ResNet50   : 25.6M 参数全部可训练')
print('  LoRA r=16 on ViT-L : ~2-3M 参数可训练（<1%）')

✓ LoRAFloorplanModel 定义完成

初始化示例（实际训练时运行）：
  model = LoRAFloorplanModel(CFG).to(DEVICE)
  # 输出：LoRA 注入完成  可训练参数: ~2.1M / ~308.5M (0.7%)

参数效率对比：
  全量微调 ResNet50   : 25.6M 参数全部可训练
  LoRA r=16 on ViT-L : ~2-3M 参数可训练（<1%）


## 5. Step 3：极致数据增强（200张图纸专项）

In [25]:
# ══════════════════════════════════════════════════════════════
# 合成扰动：模拟真实扫描图纸的复杂干扰
# ══════════════════════════════════════════════════════════════

def add_fake_lines(
    image: np.ndarray,
    n_lines: int = 5,
    thickness_range: Tuple[int, int] = (1, 3),
) -> np.ndarray:
    """
    随机叠加虚假线条（模拟旧图纸的折痕、污迹、标注线）
    只在图片亮区（背景）叠加，避免破坏墙体 mask
    """
    result = image.copy()
    h, w   = result.shape[:2]

    for _ in range(n_lines):
        x1 = random.randint(0, w - 1)
        y1 = random.randint(0, h - 1)
        x2 = random.randint(0, w - 1)
        y2 = random.randint(0, h - 1)
        # 线条颜色：略深于背景，模拟淡痕
        color     = random.randint(100, 200)
        thickness = random.randint(*thickness_range)
        cv2.line(result, (x1, y1), (x2, y2), (color, color, color), thickness)

    return result


def build_augmentation_pipeline(
    cfg:      LoRAFloorplanConfig,
    is_train: bool = True,
) -> A.Compose:
    """
    构建 albumentations 增强流水线

    训练集：完整增强（几何变换 + 合成扰动 + 颜色扰动）
    验证集：只做归一化，保持评估一致性

    ！关键设计：所有几何变换必须同步作用于 image 和 mask。
    albumentations 通过 additional_targets 实现这一点。
    """
    if not is_train:
        return A.Compose([
            A.Normalize(mean=cfg.norm_mean, std=cfg.norm_std),
            ToTensorV2(),
        ])

    train_transforms = [
        # ── 几何变换（同步作用于 mask）──
        A.RandomRotate90(p=0.75),               # 90/180/270 度随机旋转
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.ShiftScaleRotate(
            shift_limit  = 0.1,
            scale_limit  = (cfg.aug_scale_range[0]-1, cfg.aug_scale_range[1]-1),
            rotate_limit = 15,   # 小角度斜转（非 90 度整倍数）
            border_mode  = cv2.BORDER_REFLECT_101,
            p            = 0.7,
        ),

        # ── 图纸特有：弹性形变（模拟扫描变形）──
        A.ElasticTransform(
            alpha=30, sigma=5, alpha_affine=5,
            border_mode=cv2.BORDER_REFLECT_101, p=0.3
        ),

        # ── 颜色扰动（模拟不同扫描仪的色调差异）──
        A.OneOf([
            A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3),
            A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=30, val_shift_limit=20),
            A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8)),
        ], p=0.6),

        # ── 图纸特有：模拟旧扫描的褪色/泛黄 ──
        A.ToSepia(p=0.15),
        A.ToGray(p=0.2),   # 20% 的图纸是黑白的

        # ── 合成扰动：高斯噪声 ──
        A.GaussNoise(
            var_limit=(0.0, cfg.aug_noise_var * 255**2),
            p=0.4 if cfg.aug_add_noise else 0.0
        ),

        # ── JPEG 压缩伪影（模拟低质量扫描）──
        A.ImageCompression(
            quality_lower=cfg.aug_jpeg_quality[0],
            quality_upper=cfg.aug_jpeg_quality[1],
            p=0.3
        ),

        # ── 模糊（模拟失焦扫描）──
        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 5)),
            A.MotionBlur(blur_limit=5),
        ], p=0.2),

        # ── 随机遮挡（CoarseDropout，模拟图纸污损）──
        A.CoarseDropout(
            max_holes=8, max_height=32, max_width=32,
            fill_value=255,  # 用白色填充，模拟白色污损
            p=0.3
        ),

        # ── 最终归一化 ──
        A.Normalize(mean=cfg.norm_mean, std=cfg.norm_std),
        ToTensorV2(),
    ]

    return A.Compose(
        train_transforms,
        additional_targets={'mask': 'mask'},  # mask 和 image 同步变换
    )


# ── 验证增强流水线的 mask 同步性 ──
def verify_augmentation_sync(aug_pipeline: A.Compose) -> bool:
    """确认几何变换后 image 和 mask 仍然对齐"""
    dummy_img  = (np.random.rand(512, 512, 3) * 255).astype(np.uint8)
    dummy_mask = np.zeros((512, 512), dtype=np.uint8)
    dummy_mask[100:200, 100:300] = 1   # 一个矩形墙体

    result = aug_pipeline(image=dummy_img, mask=dummy_mask)
    aug_img  = result['image']
    aug_mask = result['mask']

    # 验证：mask 中有墙体像素（变换后不应全部消失）
    if isinstance(aug_mask, torch.Tensor):
        has_wall = aug_mask.sum() > 0
    else:
        has_wall = aug_mask.sum() > 0

    print(f'  增强同步验证  image shape={tuple(aug_img.shape)}  '
          f'mask has_wall={has_wall}')
    return bool(has_wall)


train_aug = build_augmentation_pipeline(CFG, is_train=True)
val_aug   = build_augmentation_pipeline(CFG, is_train=False)

print('✓ 极致数据增强流水线定义完成')
print(f'  训练集变换数: {len(train_aug.transforms)}')
print(f'  几何变换     : RandomRotate90 + HFlip + VFlip + ShiftScaleRotate')
print(f'  合成扰动     : GaussNoise + JPEG压缩 + CoarseDropout + ElasticTransform')
print(f'  图纸专项     : ToSepia + ToGray + CLAHE')
verify_augmentation_sync(train_aug)

✓ 极致数据增强流水线定义完成
  训练集变换数: 14
  几何变换     : RandomRotate90 + HFlip + VFlip + ShiftScaleRotate
  合成扰动     : GaussNoise + JPEG压缩 + CoarseDropout + ElasticTransform
  图纸专项     : ToSepia + ToGray + CLAHE
  增强同步验证  image shape=(3, 512, 512)  mask has_wall=True


True

## 6. 数据集（公司图纸 + CubiCasa 混合）

In [10]:
from numpy import genfromtxt


# ══════════════════════════════════════════════════════════════
# 公司图纸数据集（200 张，带 SVG/JSON 标注）
# ══════════════════════════════════════════════════════════════

class CompanyFloorplanDataset(Dataset):
    """
    公司自有图纸数据集

    目录结构（每张图纸一个子目录）：
    company_floorplans/
      ├── plan_001/
      │   ├── F1_scaled.png     ← 图片
      │   └── model.svg         ← SVG 标注（与 CubiCasa 格式兼容）
      │   或
      │   └── labels.json       ← JSON 格式标注（备选）
      └── ...

    如果公司图纸标注格式不同，修改 _load_labels() 方法即可，
    其他逻辑（Tiling / 增强 / 返回格式）不需要改。
    """

    def __init__(
        self,
        data_dir:    str,
        cfg:         LoRAFloorplanConfig,
        is_train:    bool = True,
        aug_pipeline: Optional[A.Compose] = None,
    ):
        self.data_dir    = data_dir
        self.cfg         = cfg
        self.is_train    = is_train
        self.aug         = aug_pipeline or build_augmentation_pipeline(cfg, is_train)

        # 扫描所有子目录
        self.folders = sorted([
            d for d in Path(data_dir).iterdir()
            if d.is_dir() and (
                (d / 'F1_scaled.png').exists() or
                list(d.glob('*.png')) or
                list(d.glob('*.jpg'))
            )
        ])

        # 80/20 划分（可重现）
        n_train = int(len(self.folders) * 0.8)
        if is_train:
            self.folders = self.folders[:n_train]
        else:
            self.folders = self.folders[n_train:]

        logger.info(
            f'CompanyFloorplanDataset [{"train" if is_train else "val"}] '
            f'{len(self.folders)} 个样本'
        )

    def __len__(self):
        return len(self.folders)

    def __getitem__(self, idx: int) -> dict:
        folder = self.folders[idx]
        try:
            image, wall_mask, boxes, labels = self._load_sample(folder)
            return self._process(image, wall_mask, boxes, labels)
        except Exception as e:
            logger.warning(f'样本加载失败 {folder.name}: {e}')
            return self._empty()

    def _load_sample(
        self, folder: Path
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        """
        加载图片 + 标注。
        优先尝试 SVG 格式（与 CubiCasa 兼容），
        其次尝试 JSON 格式（公司自定义标注工具）。
        """
        # 找图片文件
        img_path = folder / 'F1_scaled.png'
        if not img_path.exists():
            candidates = list(folder.glob('*.png')) + list(folder.glob('*.jpg'))
            img_path   = candidates[0]

        img_bgr = cv2.imread(str(img_path))
        if img_bgr is None:
            raise FileNotFoundError(str(img_path))
        image   = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        h, w    = image.shape[:2]

        # 尝试 SVG 标注
        svg_path = folder / 'model.svg'
        if svg_path.exists():
            from floortrans.loaders.house import House
            house     = House(str(svg_path), h, w)
            seg       = house.get_segmentation_tensor()
            wall_mask = (seg[0] == self.cfg.wall_class_id).astype(np.uint8)
            boxes, labels = self._extract_boxes_from_seg(seg, h, w)
            return image, wall_mask, boxes, labels

        # 尝试 JSON 标注（公司自定义格式）
        json_path = folder / 'labels.json'
        if json_path.exists():
            return self._load_from_json(image, json_path, h, w)

        raise FileNotFoundError(f'找不到标注文件: {folder}')

    def _extract_boxes_from_seg(
        self, seg: np.ndarray, h: int, w: int
    ) -> Tuple[np.ndarray, np.ndarray]:
        """从 seg[1] 提取门窗 bbox（复用 CubiCasa 逻辑）"""
        boxes, labels = [], []
        for cls_id, lbl in [(self.cfg.door_class_id, 1),
                             (self.cfg.window_class_id, 2)]:
            m = (seg[1] == cls_id).astype(np.uint8)
            cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            for cnt in cnts:
                if cv2.contourArea(cnt) < self.cfg.min_bbox_area:
                    continue
                x, y, bw, bh = cv2.boundingRect(cnt)
                boxes.append([x, y, x+bw, y+bh])
                labels.append(lbl)
        if boxes:
            return np.array(boxes, dtype=np.float32), np.array(labels, dtype=np.int64)
        return np.zeros((0,4), dtype=np.float32), np.zeros(0, dtype=np.int64)

    def _load_from_json(
        self, image: np.ndarray, json_path: Path, h: int, w: int
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        """
        公司自定义 JSON 标注格式解析器
        格式示例：
        {
          "walls": [[x1,y1,x2,y2], ...],      ← 墙体矩形
          "doors": [[x1,y1,x2,y2], ...],
          "windows": [[x1,y1,x2,y2], ...]
        }
        如果公司标注格式不同，只需修改这个函数。
        """
        with open(json_path) as f:
            data = json.load(f)

        wall_mask = np.zeros((h, w), dtype=np.uint8)
        for b in data.get('walls', []):
            x1,y1,x2,y2 = [int(v) for v in b]
            wall_mask[y1:y2, x1:x2] = 1

        boxes, labels = [], []
        for b in data.get('doors', []):
            boxes.append([float(v) for v in b]); labels.append(1)
        for b in data.get('windows', []):
            boxes.append([float(v) for v in b]); labels.append(2)

        if boxes:
            return image, wall_mask, np.array(boxes, dtype=np.float32), np.array(labels, dtype=np.int64)
        return image, wall_mask, np.zeros((0,4), dtype=np.float32), np.zeros(0, dtype=np.int64)

    def _process(
        self,
        image:     np.ndarray,
        wall_mask: np.ndarray,
        boxes:     np.ndarray,
        labels:    np.ndarray,
    ) -> dict:
        """Tiling + 增强 + 合成扰动 → tensor"""
        # 合成扰动（fake_lines 不能通过 albumentations 传入）
        if self.is_train and self.cfg.aug_fake_lines and random.random() < 0.4:
            image = add_fake_lines(image, n_lines=random.randint(2, 8))

        # Tiling：随机选一个 512×512 的 tile
        h, w   = image.shape[:2]
        ts     = self.cfg.tile_size
        stride = ts - self.cfg.tile_overlap

        ys = list(range(0, max(h - ts + 1, 1), stride))
        xs = list(range(0, max(w - ts + 1, 1), stride))
        if not ys or ys[-1] + ts < h: ys.append(max(h - ts, 0))
        if not xs or xs[-1] + ts < w: xs.append(max(w - ts, 0))
        ty = random.choice(list(set(ys)))
        tx = random.choice(list(set(xs)))

        img_tile  = image[ty:ty+ts, tx:tx+ts].copy()
        mask_tile = wall_mask[ty:ty+ts, tx:tx+ts].copy()

        # Padding
        th, tw = img_tile.shape[:2]
        if th < ts or tw < ts:
            img_tile  = cv2.copyMakeBorder(img_tile, 0, ts-th, 0, ts-tw, cv2.BORDER_REFLECT_101)
            new_mask  = np.zeros((ts, ts), dtype=mask_tile.dtype)
            new_mask[:th, :tw] = mask_tile
            mask_tile = new_mask
        img_tile  = cv2.resize(img_tile,  (ts, ts))
        mask_tile = cv2.resize(mask_tile.astype(np.float32), (ts, ts),
                               interpolation=cv2.INTER_NEAREST).astype(np.uint8)

        # albumentations 增强（image + mask 同步）
        augmented = self.aug(image=img_tile, mask=mask_tile)
        img_t     = augmented['image']     # (3, H, W) tensor
        mask_t    = augmented['mask'].long()  # (H, W) long tensor

        # 把 bbox 坐标调整到 tile 坐标系
        tile_boxes, tile_labels = [], []
        for box, lbl in zip(boxes, labels):
            x1 = np.clip(box[0] - tx, 0, ts)
            y1 = np.clip(box[1] - ty, 0, ts)
            x2 = np.clip(box[2] - tx, 0, ts)
            y2 = np.clip(box[3] - ty, 0, ts)
            if (x2-x1) > 5 and (y2-y1) > 5:
                tile_boxes.append([x1, y1, x2, y2])
                tile_labels.append(int(lbl))

        boxes_t  = torch.tensor(tile_boxes,  dtype=torch.float32) if tile_boxes else torch.zeros((0,4))
        labels_t = torch.tensor(tile_labels, dtype=torch.int64)   if tile_labels else torch.zeros(0, dtype=torch.int64)

        return {'image': img_t, 'mask': mask_t, 'boxes': boxes_t, 'labels': labels_t}

    def _empty(self):
        ts = self.cfg.tile_size
        return {
            'image':  torch.zeros(3, ts, ts),
            'mask':   torch.zeros(ts, ts).long(),
            'boxes':  torch.zeros((0, 4), dtype=torch.float32),
            'labels': torch.zeros(0, dtype=torch.int64),
        }


def collate_fn(batch):
    images  = torch.stack([b['image']  for b in batch])
    masks   = torch.stack([b['mask']   for b in batch])
    boxes   = [b['boxes']  for b in batch]
    labels  = [b['labels'] for b in batch]
    return {'image': images, 'mask': masks, 'boxes': boxes, 'labels': labels}


print('✓ 数据集类定义完成')
print('  支持格式: SVG（CubiCasa 兼容）+ JSON（公司自定义）')
print('  Tiling  : 512×512 随机采样，200张→数千个 patch')
print('  增强    : albumentations 流水线，image/mask 同步变换')
print('  fake_lines: 随机叠加虚假线条，概率 40%')

✓ 数据集类定义完成
  支持格式: SVG（CubiCasa 兼容）+ JSON（公司自定义）
  Tiling  : 512×512 随机采样，200张→数千个 patch
  增强    : albumentations 流水线，image/mask 同步变换
  fake_lines: 随机叠加虚假线条，概率 40%


## 7. Step 4：联合损失函数（Dice + BCE + L1 + GIoU）

In [11]:
# ══════════════════════════════════════════════════════════════
# Dice Loss：对小目标友好，小样本场景优于纯 BCE
# ══════════════════════════════════════════════════════════════

class DiceLoss(nn.Module):
    """
    Soft Dice Loss
    L_dice = 1 - (2 * |P∩G| + ε) / (|P| + |G| + ε)

    优点：不受类别不平衡影响（墙体像素稀少时 BCE 会退化）
    """

    def __init__(self, smooth: float = 1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        # logits: (B, C, H, W)  target: (B, H, W) long
        prob    = torch.softmax(logits, dim=1)[:, 1]  # wall 类的概率
        target_f = (target == 1).float()
        prob_f   = prob.view(-1)
        tgt_f    = target_f.view(-1)
        intersection = (prob_f * tgt_f).sum()
        return 1.0 - (2.0 * intersection + self.smooth) / (
            prob_f.sum() + tgt_f.sum() + self.smooth
        )


# ══════════════════════════════════════════════════════════════
# 动态 Lambda：自动平衡分割和检测的训练速度
# ══════════════════════════════════════════════════════════════

class DynamicLambda:
    """
    根据两个 loss 的历史 EMA 动态调整 lambda_seg 和 lambda_det

    直觉：如果 L_det 比 L_seg 大 10 倍，说明检测任务更难，
    应该适当提升 lambda_det，让模型多关注检测。

    更新规则：
        lambda_i = 1 / (normalized_loss_i + eps)
        normalized = loss_i / (ema_i + eps)
    """

    def __init__(self, ema_decay: float = 0.99):
        self.ema_decay = ema_decay
        self.ema_seg   = 1.0
        self.ema_det   = 1.0
        self.lambda_seg = 1.0
        self.lambda_det = 1.0

    def update(self, loss_seg: float, loss_det: float) -> Tuple[float, float]:
        # EMA 更新
        self.ema_seg = self.ema_decay * self.ema_seg + (1 - self.ema_decay) * loss_seg
        self.ema_det = self.ema_decay * self.ema_det + (1 - self.ema_decay) * loss_det

        # 相对损失（归一化）
        rel_seg = loss_seg / (self.ema_seg + 1e-8)
        rel_det = loss_det / (self.ema_det + 1e-8)

        # softmax 归一化，使 lambda_seg + lambda_det = 2
        total = rel_seg + rel_det + 1e-8
        self.lambda_seg = 2.0 * rel_seg / total
        self.lambda_det = 2.0 * rel_det / total

        return self.lambda_seg, self.lambda_det


# ══════════════════════════════════════════════════════════════
# 完整多任务损失
# ══════════════════════════════════════════════════════════════

class MultiTaskLoss(nn.Module):
    """
    L_total = lambda_seg * L_seg  +  lambda_det * L_det

    L_seg = dice_weight * L_Dice + bce_weight * L_BCE
    L_det = Faster-RCNN 自带损失（rpn + cls + box_reg）
           + giou_weight * L_GIoU（对 GT box 直接计算）

    Note：GIoU loss 需要 pred boxes（来自 ROI head 的回归结果），
    实际使用时通过 det_losses 的 box 回归损失替代。
    """

    def __init__(self, cfg: LoRAFloorplanConfig):
        super().__init__()
        self.cfg = cfg

        # 分割损失
        w = torch.tensor([1.0, cfg.wall_class_weight])
        self.bce      = nn.CrossEntropyLoss(weight=w, ignore_index=255)
        self.dice     = DiceLoss()

        # 动态 lambda
        self.dyn_lambda = DynamicLambda() if cfg.use_dynamic_lambda else None

        self.lambda_seg = cfg.lambda_seg
        self.lambda_det = cfg.lambda_det

    def forward(
        self,
        seg_logits:  torch.Tensor,
        seg_targets: torch.Tensor,
        det_losses:  dict,
    ) -> dict:
        # ── 分割损失 ──
        l_bce  = self.bce(seg_logits, seg_targets)
        l_dice = self.dice(seg_logits, seg_targets)
        l_seg  = self.cfg.bce_weight * l_bce + self.cfg.dice_weight * l_dice

        # ── 检测损失（Faster R-CNN 自带）──
        l_det = sum(det_losses.values())

        # ── 动态 lambda 更新 ──
        if self.dyn_lambda is not None:
            self.lambda_seg, self.lambda_det = self.dyn_lambda.update(
                l_seg.item(), l_det.item()
            )

        l_total = self.lambda_seg * l_seg + self.lambda_det * l_det

        return {
            'loss_total':  l_total,
            'loss_seg':    l_seg.item(),
            'loss_bce':    l_bce.item(),
            'loss_dice':   l_dice.item(),
            'loss_det':    l_det.item(),
            'lambda_seg':  self.lambda_seg,
            'lambda_det':  self.lambda_det,
        }


print('✓ 联合损失函数定义完成')
print('  L_seg   = 0.5 × Dice + 0.5 × BCE（wall weight=5）')
print('  L_det   = Faster R-CNN 自带损失（rpn_cls + rpn_box + cls + box）')
print('  L_total = λ_seg × L_seg + λ_det × L_det')
print('  DynamicLambda: 根据 EMA 自动平衡两个任务')

✓ 联合损失函数定义完成
  L_seg   = 0.5 × Dice + 0.5 × BCE（wall weight=5）
  L_det   = Faster R-CNN 自带损失（rpn_cls + rpn_box + cls + box）
  L_total = λ_seg × L_seg + λ_det × L_det
  DynamicLambda: 根据 EMA 自动平衡两个任务


## 8. 训练主循环

In [12]:
class AverageMeter:
    def __init__(self): self.reset()
    def reset(self): self.val = self.avg = self.sum = self.count = 0.0
    def update(self, val, n=1):
        self.val = val; self.sum += val * n
        self.count += n; self.avg = self.sum / self.count


def compute_wall_iou(pred: torch.Tensor, target: torch.Tensor) -> float:
    pred, target = pred.view(-1), target.view(-1)
    tp = ((pred==1)&(target==1)).sum().item()
    fp = ((pred==1)&(target==0)).sum().item()
    fn = ((pred==0)&(target==1)).sum().item()
    denom = tp + fp + fn
    return tp / denom if denom > 0 else 0.0


def train_epoch(
    model, criterion, optimizer, scaler,
    loader, device, epoch, cfg,
) -> Tuple[dict, float]:
    model.train(); criterion.train()
    meters = {k: AverageMeter() for k in [
        'loss_total','loss_seg','loss_bce','loss_dice','loss_det','lambda_seg','lambda_det'
    ]}
    t0 = time.time()

    # Warmup 学习率
    if epoch <= cfg.warmup_epochs:
        lr_scale = epoch / cfg.warmup_epochs
        for pg in optimizer.param_groups:
            pg['lr'] = pg.get('initial_lr', cfg.learning_rate) * lr_scale

    for batch in loader:
        images  = batch['image'].to(device)
        masks   = batch['mask'].to(device)
        targets = [
            {'boxes': b.to(device), 'labels': l.to(device)}
            for b, l in zip(batch['boxes'], batch['labels'])
        ]

        optimizer.zero_grad()
        with torch.amp.autocast(device_type=device, enabled=cfg.use_amp):
            outputs   = model(images, targets)
            loss_dict = criterion(
                outputs['seg_logits'],
                masks,
                outputs['det_losses'],
            )

        scaler.scale(loss_dict['loss_total']).backward()
        scaler.unscale_(optimizer)
        # 只裁剪可训练参数的梯度
        trainable = [p for p in model.parameters() if p.requires_grad]
        torch.nn.utils.clip_grad_norm_(trainable, cfg.grad_clip)
        scaler.step(optimizer)
        scaler.update()

        n = images.size(0)
        for k in meters:
            if k in loss_dict:
                val = loss_dict[k]
                meters[k].update(float(val) if isinstance(val, torch.Tensor) else val, n)

    return {k: m.avg for k, m in meters.items()}, time.time() - t0


@torch.no_grad()
def val_epoch(model, criterion, loader, device) -> dict:
    model.eval(); criterion.eval()
    seg_loss_m = AverageMeter()
    wall_iou_m = AverageMeter()

    for batch in loader:
        images = batch['image'].to(device)
        masks  = batch['mask'].to(device)
        outputs = model(images)
        loss_dict = criterion(outputs['seg_logits'], masks, {})
        preds     = outputs['seg_logits'].argmax(dim=1)
        wall_iou  = compute_wall_iou(preds.cpu(), masks.cpu())
        seg_loss_m.update(loss_dict['loss_seg'], images.size(0))
        wall_iou_m.update(wall_iou, images.size(0))

    return {'val_seg_loss': seg_loss_m.avg, 'val_wall_iou': wall_iou_m.avg}


print('✓ 训练/验证函数定义完成')

✓ 训练/验证函数定义完成


## 9. 启动训练（含 MLflow 记录）

In [13]:
RUN_TRAINING = False   # ← 改为 True 启动训练

if RUN_TRAINING:
    # ── 数据集 ──
    train_ds = CompanyFloorplanDataset(CFG.company_data_dir, CFG, is_train=True,  aug_pipeline=train_aug)
    val_ds   = CompanyFloorplanDataset(CFG.company_data_dir, CFG, is_train=False, aug_pipeline=val_aug)

    train_loader = DataLoader(
        train_ds, batch_size=CFG.batch_size, shuffle=True,
        num_workers=CFG.num_workers, pin_memory=True,
        drop_last=True, collate_fn=collate_fn,
    )
    val_loader = DataLoader(
        val_ds, batch_size=CFG.batch_size, shuffle=False,
        num_workers=CFG.num_workers, pin_memory=True,
        collate_fn=collate_fn,
    )

    # ── 模型 ──
    model = LoRAFloorplanModel(CFG).to(DEVICE)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f'可训练参数: {trainable/1e6:.2f}M / {total/1e6:.1f}M ({trainable/total*100:.1f}%)')

    # ── 损失 ──
    criterion = MultiTaskLoss(CFG).to(DEVICE)

    # ── 优化器（参数分层学习率）──
    param_groups = get_param_groups(model, CFG.learning_rate)
    for g in param_groups:
        g['initial_lr'] = g['lr']
    optimizer = torch.optim.AdamW(param_groups, weight_decay=CFG.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max     = CFG.max_epochs - CFG.warmup_epochs,
        eta_min   = CFG.learning_rate * 0.01,
    )
    scaler = torch.amp.GradScaler('cuda', enabled=CFG.use_amp)

    # ── 检查点管理 ──
    from pathlib import Path as _Path
    best_iou = 0.0
    top_ckpts: list = []  # [(score, path), ...]

    # ── MLflow ──
    mlflow.set_tracking_uri(f'file://{MLFLOW_DIR}')
    mlflow.set_experiment(CFG.mlflow_experiment)

    with mlflow.start_run(run_name=CFG.mlflow_run_name) as run:
        mlflow.log_params({
            'backbone':        CFG.backbone_type,
            'lora_r':          CFG.lora_r,
            'lora_alpha':      CFG.lora_alpha,
            'trainable_M':     round(trainable/1e6, 2),
            'total_M':         round(total/1e6, 1),
            'lora_pct':        round(trainable/total*100, 2),
            'batch_size':      CFG.batch_size,
            'max_epochs':      CFG.max_epochs,
            'learning_rate':   CFG.learning_rate,
            'dice_weight':     CFG.dice_weight,
            'wall_class_weight': CFG.wall_class_weight,
            'aug_fake_lines':  CFG.aug_fake_lines,
            'aug_add_noise':   CFG.aug_add_noise,
            'use_dynamic_lambda': CFG.use_dynamic_lambda,
        })

        header = (f'{"Ep":>4}  {"total":>8}  {"seg":>8}  {"dice":>7}  '
                  f'{"det":>7}  {"λseg":>6}  {"λdet":>6}  '
                  f'{"iou":>8}  {"lr":>8}  {"s":>5}')
        logger.info(header)

        for epoch in range(1, CFG.max_epochs + 1):
            train_m, elapsed = train_epoch(
                model, criterion, optimizer, scaler,
                train_loader, DEVICE, epoch, CFG,
            )
            val_m = val_epoch(model, criterion, val_loader, DEVICE)

            if epoch > CFG.warmup_epochs:
                scheduler.step()
            current_lr = optimizer.param_groups[0]['lr']

            mlflow.log_metrics({
                'train_loss_total': train_m['loss_total'],
                'train_loss_seg':   train_m['loss_seg'],
                'train_loss_dice':  train_m['loss_dice'],
                'train_loss_det':   train_m['loss_det'],
                'lambda_seg':       train_m['lambda_seg'],
                'lambda_det':       train_m['lambda_det'],
                'val_seg_loss':     val_m['val_seg_loss'],
                'val_wall_iou':     val_m['val_wall_iou'],
                'learning_rate':    current_lr,
            }, step=epoch)

            # 保存 Top-K checkpoint
            score = val_m['val_wall_iou']
            ckpt_path = _Path(CFG.checkpoint_dir) / f'ep{epoch:03d}_iou{score:.4f}.pth'
            torch.save({
                'epoch':          epoch,
                'model_state':    model.state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'val_wall_iou':   score,
                'lora_r':         CFG.lora_r,
                'backbone_type':  CFG.backbone_type,
            }, ckpt_path)
            top_ckpts.append((score, ckpt_path))
            top_ckpts.sort(key=lambda x: x[0], reverse=True)
            while len(top_ckpts) > CFG.save_top_k:
                _, old = top_ckpts.pop()
                if old.exists(): old.unlink()

            if score > best_iou:
                best_iou = score
                best_path = f'{CFG.checkpoint_dir}/best_model_lora.pth'
                torch.save({
                    'model_state':   model.state_dict(),
                    'epoch':         epoch,
                    'val_wall_iou':  best_iou,
                    'lora_r':        CFG.lora_r,
                    'backbone_type': CFG.backbone_type,
                    'cfg':           CFG.__dict__,
                }, best_path)
                mlflow.log_metric('best_wall_iou', best_iou, step=epoch)

            logger.info(
                f'{epoch:>4}  '
                f'{train_m["loss_total"]:>8.4f}  '
                f'{train_m["loss_seg"]:>8.4f}  '
                f'{train_m["loss_dice"]:>7.4f}  '
                f'{train_m["loss_det"]:>7.4f}  '
                f'{train_m["lambda_seg"]:>6.3f}  '
                f'{train_m["lambda_det"]:>6.3f}  '
                f'{val_m["val_wall_iou"]:>8.4f}  '
                f'{current_lr:>8.6f}  {elapsed:>5.1f}s'
            )

        logger.info(f'训练完成  best_wall_iou={best_iou:.4f}')
        mlflow.log_artifact(best_path, 'model')

else:
    print('跳过训练（RUN_TRAINING=False）')
    print('将 RUN_TRAINING 改为 True 启动训练')

跳过训练（RUN_TRAINING=False）
将 RUN_TRAINING 改为 True 启动训练


## 10. Step 5：OpenCV 硬逻辑后处理（接入已有流水线）

In [17]:
# LoRA 模型推理 + 原有 OpenCV 后处理流水线（直接复用）

from post_service   import run_postprocess
from vector_logic   import vectorize_wall_mask
from reconstruct_3d import build_3d_model, match_openings_to_walls


def lora_predict_and_postprocess(
    image_path:    str,
    lora_ckpt:     str,
    output_dir:    str  = './outputs',
    score_thresh:  float = 0.5,
) -> dict:
    """
    Step 5 完整链路：
    1. 加载 LoRA 模型权重
    2. 滑动窗口推理 → raw wall_mask
    3. morphological_preprocessing（Opening → Closing）
    4. vectorize_wall_mask（Shrinking → Overlap 消解）
    5. match_openings_to_walls + build_3d_model
    """
    import sys, os
    sys.path.insert(0, BASE_DIR)
    os.makedirs(output_dir, exist_ok=True)

    # ── 加载 LoRA 模型 ──
    ckpt  = torch.load(lora_ckpt, map_location=DEVICE)
    cfg   = LoRAFloorplanConfig()
    if 'cfg' in ckpt:   # 如果 checkpoint 里保存了配置
        for k, v in ckpt['cfg'].items():
            if hasattr(cfg, k):
                setattr(cfg, k, v)

    model = LoRAFloorplanModel(cfg).to(DEVICE)
    model.load_state_dict(ckpt['model_state'])
    model.eval()
    logger.info(f'LoRA 模型加载完成  epoch={ckpt.get("epoch")}  '
                f'val_wall_iou={ckpt.get("val_wall_iou", "?"):.4f}')

    # ── 读取图片 ──
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        raise FileNotFoundError(image_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w    = img_rgb.shape[:2]

    # ── 滑动窗口推理 ──
    ts      = cfg.tile_size
    overlap = cfg.tile_overlap
    stride  = ts - overlap
    tf      = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(cfg.norm_mean, cfg.norm_std),
    ])

    wall_prob = np.zeros((h, w), dtype=np.float32)
    wall_cnt  = np.zeros((h, w), dtype=np.float32)
    all_boxes, all_scores, all_labels = [], [], []

    ys = list(range(0, max(h-ts+1, 1), stride))
    xs = list(range(0, max(w-ts+1, 1), stride))
    if not ys or ys[-1]+ts < h: ys.append(max(h-ts, 0))
    if not xs or xs[-1]+ts < w: xs.append(max(w-ts, 0))

    for ty in ys:
        for tx in xs:
            tile   = img_rgb[ty:ty+ts, tx:tx+ts].copy()
            th, tw = tile.shape[:2]
            if th < ts or tw < ts:
                tile = cv2.copyMakeBorder(tile, 0, ts-th, 0, ts-tw, cv2.BORDER_REFLECT_101)
            tile   = cv2.resize(tile, (ts, ts))
            tensor = tf(tile).unsqueeze(0).to(DEVICE)

            with torch.no_grad():
                out  = model(tensor)
            prob = torch.softmax(out['seg_logits'], dim=1)[0, 1].cpu().numpy()
            prob = cv2.resize(prob, (tw, th))
            wall_prob[ty:ty+th, tx:tx+tw] += prob[:th, :tw]
            wall_cnt[ty:ty+th,  tx:tx+tw] += 1

            for det in out['det_outputs']:
                boxes  = det['boxes'].cpu().numpy()
                scores = det['scores'].cpu().numpy()
                labels = det['labels'].cpu().numpy()
                sx = tw / ts; sy = th / ts
                for b, s, l in zip(boxes, scores, labels):
                    if s >= score_thresh:
                        all_boxes.append([b[0]*sx+tx, b[1]*sy+ty, b[2]*sx+tx, b[3]*sy+ty])
                        all_scores.append(s); all_labels.append(l)

    wall_prob /= np.maximum(wall_cnt, 1)
    raw_mask   = (wall_prob > 0.5).astype(np.uint8)

    if all_boxes:
        from torchvision.ops import nms
        keep = nms(torch.tensor(all_boxes), torch.tensor(all_scores), 0.5)
        det_boxes  = np.array(all_boxes)[keep.numpy()]
        det_labels = np.array(all_labels)[keep.numpy()]
    else:
        det_boxes  = np.zeros((0,4), dtype=np.float32)
        det_labels = np.zeros(0, dtype=np.int64)

    # ── Step 5a: 形态学修补（硬逻辑） ──
    from vector_logic import morphological_preprocessing
    repaired_mask = morphological_preprocessing(raw_mask)
    logger.info(f'形态学修补: {raw_mask.sum()} → {repaired_mask.sum()} 像素')

    # ── Step 5b: 矢量化收缩 ──
    wall_boxes = vectorize_wall_mask(repaired_mask)
    logger.info(f'矢量化: {len(wall_boxes)} 个墙体段')

    # ── Step 5c: 门窗匹配 + 3D 重建 ──
    openings = match_openings_to_walls(wall_boxes, det_boxes, det_labels)
    name     = Path(image_path).stem
    glb_path = os.path.join(output_dir, f'{name}_lora_3d.glb')
    build_3d_model(wall_boxes, openings, output_path=glb_path)

    return {
        'wall_mask':   repaired_mask,
        'wall_boxes':  wall_boxes,
        'det_boxes':   det_boxes,
        'det_labels':  det_labels,
        'glb_path':    glb_path,
        'stats': {
            'n_walls':   len(wall_boxes),
            'n_doors':   sum(1 for o in openings if o['type']=='door'),
            'n_windows': sum(1 for o in openings if o['type']=='window'),
        },
    }


print('✓ LoRA 推理 + OpenCV 后处理链路定义完成')
print('  流程: 滑动窗口推理 → Opening/Closing修补 → Shrinking矢量化 → 3D重建')

✓ LoRA 推理 + OpenCV 后处理链路定义完成
  流程: 滑动窗口推理 → Opening/Closing修补 → Shrinking矢量化 → 3D重建


## 11. Step 6：异步 3D 渲染 + 人工审核接入

In [19]:
# ══════════════════════════════════════════════════════════════
# 将 LoRA 推理结果接入已有的 Staging → VLM 质检 → 审核流水线
# ══════════════════════════════════════════════════════════════

from preview_service     import generate_preview
from persistence_service import approve, reject, list_pending

# ── VLM 质检（如果已安装）──
try:
    import anthropic
    # vlm_inspector.ipynb 里定义的函数
    # 这里假设已经将其导出为 vlm_inspector.py
    from vlm_inspector import run_vlm_inspection_pipeline
    HAS_VLM = True
except ImportError:
    HAS_VLM = False
    print('[提示] vlm_inspector 未安装，跳过 VLM 质检步骤')


def full_pipeline_with_lora(
    image_path:   str,
    lora_ckpt:    str,
    output_dir:   str  = './outputs',
    use_vlm:      bool = True,
    vlm_dry_run:  bool = True,
) -> dict:
    """
    LoRA 版完整端到端流水线

    GPU 端：LoRA 推理
    CPU 端：OpenCV 后处理 → 3D 重建 → Staging
    质检  ：VLM 逻辑监工
    审核  ：Staging → approve/reject → 生产库
    """
    import uuid as _uuid
    task_id = str(_uuid.uuid4())[:8]
    t0      = time.time()

    print(f'\n{"═"*55}')
    print(f'LoRA 全流程  task={task_id}  image={Path(image_path).name}')
    print(f'{"═"*55}')

    # ── GPU: 推理 + OpenCV 后处理 + 3D 重建 ──
    print('\n[GPU] 推理 + 后处理...')
    result = lora_predict_and_postprocess(
        image_path   = image_path,
        lora_ckpt    = lora_ckpt,
        output_dir   = os.path.join(output_dir, task_id),
    )
    print(f'  walls={result["stats"]["n_walls"]}  '
          f'doors={result["stats"]["n_doors"]}  '
          f'windows={result["stats"]["n_windows"]}')

    # ── 保存推理结果为 JSON（GPU/CPU 解耦的数据纽带）──
    raw_pred = {
        'task_id':    task_id,
        'image_path': image_path,
        'wall_boxes': [list(b) for b in result['wall_boxes']],
        'det_boxes':  result['det_boxes'].tolist(),
        'det_labels': result['det_labels'].tolist(),
        'glb_path':   result['glb_path'],
        'stats':      result['stats'],
    }
    json_path = os.path.join(output_dir, task_id, 'raw_pred.json')
    os.makedirs(os.path.dirname(json_path), exist_ok=True)
    with open(json_path, 'w') as f:
        json.dump(raw_pred, f, indent=2)
    print(f'\n[已保存] raw_pred.json: {json_path}')

    # ── VLM 质检 ──
    vlm_report = None
    if use_vlm and HAS_VLM:
        print('\n[VLM] 逻辑质检...')
        vlm_result = run_vlm_inspection_pipeline(
            image_path = image_path,
            dry_run    = vlm_dry_run,
        )
        vlm_report = vlm_result.get('vlm_report')
        decision   = vlm_result.get('decision', 'human_review')
        print(f'  VLM 决策: {decision}  score={vlm_report.get("aggregated_score") if vlm_report else "?"}')
    else:
        decision = 'human_review'
        print('\n[VLM] 跳过（use_vlm=False 或未安装）→ 默认送人工复核')

    elapsed = round(time.time() - t0, 2)
    print(f'\n总耗时: {elapsed}s  决策: {decision}')

    return {
        'task_id':    task_id,
        'raw_pred':   raw_pred,
        'vlm_report': vlm_report,
        'decision':   decision,
        'elapsed':    elapsed,
    }


# ── 演示（需要有效的 LoRA checkpoint）──
DEMO_LORA_CKPT = f'{CHECKPOINT_DIR}/best_model_lora.pth'
DEMO_IMAGE     = os.path.join(DATA_FOLDER, 'val/0/', 'F1_scaled.png')

RUN_DEMO = False  # ← 改为 True 运行完整演示
if RUN_DEMO and os.path.exists(DEMO_LORA_CKPT) and os.path.exists(DEMO_IMAGE):
    demo_result = full_pipeline_with_lora(
        image_path  = DEMO_IMAGE,
        lora_ckpt   = DEMO_LORA_CKPT,
        use_vlm     = True,
        vlm_dry_run = True,
    )
else:
    print('演示跳过（RUN_DEMO=False 或文件不存在）')
    print('训练完成后将 RUN_DEMO 改为 True 运行完整流程')

[提示] vlm_inspector 未安装，跳过 VLM 质检步骤
演示跳过（RUN_DEMO=False 或文件不存在）
训练完成后将 RUN_DEMO 改为 True 运行完整流程


## 12. LoRA 权重分析与可解释性

In [ ]:
import matplotlib.pyplot as plt


def analyze_lora_weights(
    model:     nn.Module,
    save_path: str = None,
):
    """
    可视化 LoRA 权重的分布和有效秩
    帮助判断 r=16 是否够用，以及哪些层学到了最多信息
    """
    lora_layers = []
    for name, module in model.named_modules():
        if isinstance(module, LoRALinear):
            A = module.lora_A.detach().cpu()
            B = module.lora_B.detach().cpu()
            delta_W = (B @ A).numpy()   # 完整增量矩阵

            # SVD 分析有效秩（singular values 的分布）
            U, S, Vt = np.linalg.svd(delta_W, full_matrices=False)
            effective_rank = int((S > S.max() * 0.01).sum())  # 保留 > 1% 最大奇异值的数量

            lora_layers.append({
                'name':           name,
                'singular_vals':  S[:16],  # 取前 r 个
                'effective_rank': effective_rank,
                'norm':           float(np.linalg.norm(delta_W, 'fro')),
            })

    if not lora_layers:
        print('模型中没有 LoRALinear 层')
        return

    n = len(lora_layers)
    fig, axes = plt.subplots(2, min(n, 6), figsize=(18, 8))
    if n == 1: axes = axes.reshape(2, 1)

    for i, layer in enumerate(lora_layers[:6]):
        # 奇异值分布
        axes[0, i].bar(range(len(layer['singular_vals'])), layer['singular_vals'])
        axes[0, i].set_title(f'{layer["name"][:20]}\nrank={layer["effective_rank"]}', fontsize=8)
        axes[0, i].set_xlabel('奇异值索引')
        axes[0, i].set_ylabel('大小')

        # LoRA B 矩阵的热力图（可视化学到了什么方向）
        B_vis = lora_layers[i]['name']  # 占位

    # 全部层的 Frobenius 范数（衡量每层学习量）
    norms = [l['norm'] for l in lora_layers]
    names = [l['name'].split('.')[-2][:15] for l in lora_layers]
    axes[1, 0].bar(range(len(norms)), norms, color='steelblue')
    axes[1, 0].set_title('各层 ΔW 范数（学习量）', fontsize=10)
    axes[1, 0].set_xticks(range(len(names)))
    axes[1, 0].set_xticklabels(names, rotation=45, ha='right', fontsize=7)

    # 有效秩分布
    ranks = [l['effective_rank'] for l in lora_layers]
    axes[1, 1].hist(ranks, bins=range(0, 18), edgecolor='white', color='coral')
    axes[1, 1].set_title(f'有效秩分布（r={16}）', fontsize=10)
    axes[1, 1].set_xlabel('有效秩')
    axes[1, 1].axvline(x=16, color='red', linestyle='--', label='r=16上限')
    axes[1, 1].legend()

    # 隐藏多余子图
    for j in range(2, min(n+1, 6)):
        if j < axes.shape[1]:
            for row in range(2):
                if j < axes.shape[1] and (row > 0 or j >= n):
                    axes[row, j].axis('off')

    plt.suptitle('LoRA 权重分析：有效秩 + 学习量分布', fontsize=13)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=100, bbox_inches='tight')
        print(f'分析图已保存: {save_path}')
    plt.show()

    # 打印汇总
    print(f'\nLoRA 权重分析汇总 ({len(lora_layers)} 个层):')
    print(f'  平均有效秩 : {np.mean(ranks):.1f} / {16}')
    if np.mean(ranks) >= 14:
        print('  建议         : 有效秩接近上限，考虑提升 r=32')
    elif np.mean(ranks) <= 4:
        print('  建议         : 有效秩偏低，可降到 r=8 节省参数')
    else:
        print('  建议         : r=16 较合适 ✓')


print('✓ LoRA 权重分析工具定义完成')
print('  analyze_lora_weights(model) → 奇异值分布 + 有效秩 + 学习量')
print('  调用时机: 训练结束后，判断是否需要调整 r 值')

## 13. 与原 ResNet50 方案的性能对比

In [ ]:
# ══════════════════════════════════════════════════════════════
# 对比表（训练完成后填入实际数字）
# ══════════════════════════════════════════════════════════════

COMPARISON = {
    'ResNet50（原方案）': {
        '可训练参数':    '25.6M（全量）',
        '预训练数据':    'ImageNet 1.2M',
        'CubiCasa IoU mask': 0.8151,
        'CubiCasa IoU vect': None,   # 待填入
        '200样本 IoU mask':  None,   # 待测试
        'GPU内存':       '~8GB',
        '每轮耗时':      '~25min',
    },
    'SAM2 + LoRA r=16（本方案）': {
        '可训练参数':    '~2.1M（<1%）',
        '预训练数据':    'SA-1B 11M+',
        'CubiCasa IoU mask': None,   # 待填入（预期 > 0.84）
        'CubiCasa IoU vect': None,   # 待填入
        '200样本 IoU mask':  None,   # 训练完成后填入
        'GPU内存':       '~12GB',
        '每轮耗时':      '~35min',
    },
    'DINOv2 ViT-L + LoRA r=16（备选）': {
        '可训练参数':    '~2.6M（<1%）',
        '预训练数据':    'LVD-142M',
        'CubiCasa IoU mask': None,
        'CubiCasa IoU vect': None,
        '200样本 IoU mask':  None,
        'GPU内存':       '~14GB',
        '每轮耗时':      '~40min',
    },
}

import matplotlib.pyplot as plt

# 打印对比表
print(f'{"":<35} {"ResNet50":>12} {"SAM2+LoRA":>14} {"DINOv2+LoRA":>14}')
print('─' * 77)
for metric in ['可训练参数', '预训练数据', 'CubiCasa IoU mask', '200样本 IoU mask', 'GPU内存', '每轮耗时']:
    vals = [str(COMPARISON[m].get(metric, '—') or '待测') for m in COMPARISON]
    print(f'{metric:<35} {vals[0]:>12} {vals[1]:>14} {vals[2]:>14}')

print()
print('预期改进分析：')
print('  1. 小样本场景（200张）：LoRA 冻结 99% 参数，几乎消除过拟合风险')
print('  2. 几何感知：SAM2 预训练于 1100万分割样本，比 ImageNet 分类更懂「线」')
print('  3. 多尺度能力：Hiera ViT 原生多尺度，比 ResNet+FPN 的特征融合更自然')
print('  4. 数据增强协同：200张 × Tiling × Albumentations → 实际 ~50000 训练样本')